# 코드 설명

- **주제** : 악성코드 탐지 모델(MalConv)을 상대로 **적대적 예제**를 만들어, 딥러닝 탐지기의 강건성을 평가하기 (방어·연구 목적)
- **방법** : 파일 끝에 덧붙인 패딩 바이트만 FGSM(gradient ascent)으로 조금씩 바꿔, 탐지 점수를 "정상" 쪽으로 밀어붙임. 원본 실행 코드 영역은 건드리지 않음
- **사용 파일**
    - `MalGAN_input/malconv.h5` — 사전학습된 MalConv 탐지 모델 (저장소에 포함, 약 12MB)
    - `MalGAN_input/samplesIn.csv` — 입력 샘플 목록
    - 보조 모듈 : `MalGAN_preprocess.py`, `MalGAN_utils.py`, `MalGAN_gen_adv_examples.py`

> **⚠ 입력 샘플 안내 (중요)**
>
> 기본 `samplesIn.csv` 는 정상 PE 샘플(원본/UPX/Amber) 30개와 **악성 PE 샘플(`Malicious PE Samples/`) 10개**를 가리킵니다.
>
> - **악성 샘플 폴더(`Malicious PE Samples/`)는 저장소·Release에 포함하지 않습니다.** 악성코드이므로 공개 배포하지 않으며, 실습자가 **직접 확보해 로컬에 두어야** 합니다.
> - 그 폴더가 없는 환경(공개 Colab 등)에서는 **악성 샘플 행이 자동으로 건너뛰어집니다**(`[건너뜀] 파일 없음: ...`). 정상 샘플 30개만으로도 파이프라인 전체(모델 로드 → 전처리 → 임베딩 → FGSM → 최근접 바이트 탐색 → 저장)가 동작합니다.
> - 교재 원본의 입력 목록(해시 형태)은 `MalGAN_input/samplesIn_original.csv` 에 보존해 두었습니다.

> **최신화 메모 (TF1 → TF2 / Keras 3)**
>
> | 원본 (TF1, 동작 안 함) | 현재 (TF2) | 위치 |
> |---|---|---|
> | `keras.backend.tensorflow_backend.set_session` | 제거 | `MalGAN_utils.py` |
> | `tf.ConfigProto` / `tf.Session` | `tf.config` (GPU 메모리 growth) | `limit_gpu_memory` |
> | `K.gradients(loss, …)` | `tf.GradientTape` | `fgsm` |
> | `K.function([input], [emb])` | `keras.Model(input, emb)` 서브모델 | `build_emb_models` |
> | `keras.preprocessing.sequence.pad_sequences` | `keras.utils.pad_sequences` | `MalGAN_preprocess.py` |
>
> **malconv.h5 로드** : 이 `.h5` 는 구버전 Keras로 저장되어 최신 Keras 3에서는 바로 열리지 않습니다. `TF_USE_LEGACY_KERAS=1` 환경변수 + `tf-keras` 호환 패키지를 쓰면 구형 모델을 그대로 로드할 수 있습니다(이 환경변수는 **반드시 `tensorflow` 를 import 하기 전에** 설정해야 합니다).
>
> **MalConv 구조 주의** : 임베딩 뒤가 두 conv 분기를 곱하는 구조라, 레이어를 순서대로 다시 쌓는 방식으로는 서브모델을 만들 수 없습니다. 그래서 함수형 API로 중간 텐서(`emb_layer.output`)를 입력으로 지정해 `임베딩→출력` 서브모델을 잘라냈습니다.

In [1]:
# ── Google Colab 준비 셀 ─────────────────────────────────────────
# ★ malconv.h5 로드를 위해 tensorflow import 전에 환경변수를 먼저 설정합니다.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import subprocess
import sys

# PE 샘플(정상 파일, 약 440MB)을 올려둔 곳의 zip 직접 다운로드 주소.
# 비워 두면 다운로드를 건너뜁니다. (malconv.h5 는 저장소에 포함되어 있어 받을 필요 없음)
데이터_URL = "https://github.com/isl-hedgehog/python-cyber-seminar/releases/download/pe-samples-v1/pe_samples.zip"
데이터폴더 = ["Benign PE Samples", "Benign PE Samples UPX", "Benign PE Samples Amber"]

if "google.colab" in sys.modules:
    저장소 = "https://github.com/isl-hedgehog/python-cyber-seminar.git"
    이름 = 저장소.rsplit("/", 1)[-1].removesuffix(".git")
    실습폴더 = "02-adversarial-malware"
    if not os.path.isdir(f"/content/{이름}"):
        subprocess.run(["git", "clone", "-q", 저장소], cwd="/content", check=True)
    os.chdir(f"/content/{이름}/{실습폴더}")
    # Colab의 기본 TensorFlow는 Keras 3를 쓰므로, 구형 h5 로드를 위해 tf-keras를 설치합니다.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf-keras"], check=True)
    print("Colab 준비 완료 :", os.getcwd())
else:
    print("로컬 환경 — 준비 셀을 건너뜁니다.")

# 입력 대역으로 쓸 정상 PE 샘플이 있는지 확인
if not os.path.isdir("Benign PE Samples"):
    print("⚠ 'Benign PE Samples' 폴더가 없습니다. 데이터_URL 을 설정하거나, 폴더를 이 위치에 두세요.")
else:
    print("입력 샘플 폴더 확인됨.")

로컬 환경 — 준비 셀을 건너뜁니다.
입력 샘플 폴더 확인됨.


### 1. 모듈 불러오기

`TF_USE_LEGACY_KERAS=1` 이 설정된 상태에서 `tensorflow.keras` 를 불러오면 구형 h5 를 읽을 수 있는 호환(tf-keras) 백엔드가 사용됩니다. 보조 모듈 3개도 함께 불러옵니다.

In [2]:
# 안전장치: 준비 셀을 건너뛰고 이 셀부터 실행하는 경우를 대비해 한 번 더 설정
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import pandas as pd
from tensorflow import keras

import MalGAN_utils
import MalGAN_gen_adv_examples

### 2. 설정값

- `pad_percent` : 파일 길이의 몇 %까지 패딩 바이트를 덧붙여 조작할지 (0.1 = 10%)
- `threshold` : 이 값보다 탐지 점수가 낮으면(=악성으로 판단하면) 공격을 시도 (0→악성, 1→정상)
- `step_size` : FGSM 한 걸음의 크기
- `limit` : GPU 메모리 제한 비율 (0 이면 제한하지 않음)

In [3]:
save_path = "MalGAN_output"
model_path = "MalGAN_input/malconv.h5"
log_path = "MalGAN_output/adversarial_log.csv"
pad_percent = 0.1
threshold = 0.6
step_size = 0.01
limit = 0.0
input_samples = "MalGAN_input/samplesIn.csv"

os.makedirs(save_path, exist_ok=True)
MalGAN_utils.limit_gpu_memory(limit)

### 3. 모델 로드와 적대적 예제 생성

입력 목록을 읽어 MalConv 모델을 로드한 뒤, 파일마다 적대적 예제를 생성합니다. 각 파일의 로그(원본 점수, 패딩 길이, 공격 후 점수)가 출력되고, 결과 바이트는 `MalGAN_output/` 에 저장됩니다.

- 원본 점수가 `threshold` 보다 **낮은**(악성으로 분류된) 파일만 공격 대상이 됩니다. 그 외에는 원본 바이트를 그대로 저장합니다.
- 정상 대역 샘플을 쓰므로, 점수가 높은 파일은 "공격 불필요"로 건너뛰는 동작을 함께 확인할 수 있습니다.

In [4]:
df = pd.read_csv(input_samples, header=None)
fn_list = df[0].values
model = keras.models.load_model(model_path)

adv_samples, log = MalGAN_gen_adv_examples.gen_adv_samples(
    model, fn_list, pad_percent, step_size, threshold
)

log.save(log_path)
# gen_adv_samples 는 (파일명, 바이트) 쌍을 돌려준다(없는 파일은 건너뛰므로 fn_list 와 길이가 다를 수 있음).
for fn, adv in adv_samples:
    _fn = fn.split("/")[-1]
    dst = os.path.join(save_path, _fn)
    print(dst)
    with open(dst, "wb") as f:
        f.write(adv)


FILE: Benign PE Samples/ARP.EXE
	file length: 22528
	pad length: 2252
	loss: nan
	score: nan
	original score: 0.6124774217605591



FILE: Benign PE Samples/AtBroker.exe
	file length: 49152
	pad length: 4915
	loss: 0.039518583565950394
	score: 0.039518583565950394
	original score: 0.039518583565950394

FILE: Benign PE Samples/BackgroundTransferHost.exe
	file length: 34816
	pad length: 3481
	loss: nan
	score: nan
	original score: 0.9946382641792297

FILE: Benign PE Samples/ByteCodeGenerator.exe
	file length: 48128
	pad length: 4812
	loss: nan
	score: nan
	original score: 0.994793713092804

FILE: Benign PE Samples/CameraSettingsUIHost.exe
	file length: 29464
	pad length: 2946
	loss: nan
	score: nan
	original score: 0.9999725222587585

FILE: Benign PE Samples/CheckNetIsolation.exe
	file length: 26112
	pad length: 2611
	loss: nan
	score: nan
	original score: 0.9430158734321594

FILE: Benign PE Samples/CloudNotifications.exe
	file length: 66728
	pad length: 6672
	loss: nan
	score: nan
	original score: 0.9789838790893555



FILE: Benign PE Samples/ComputerDefaults.exe
	file length: 61440
	pad length: 6144
	loss: 0.014175438322126865
	score: 0.014175438322126865
	original score: 0.014175438322126865

FILE: Benign PE Samples/HOSTNAME.EXE
	file length: 11776
	pad length: 1177
	loss: nan
	score: nan
	original score: 0.9994351863861084

FILE: Benign PE Samples/InfDefaultInstall.exe
	file length: 11776
	pad length: 1177
	loss: 0.9996917843818665
	score: 1.0
	original score: 0.30977413058280945



FILE: Benign PE Samples Amber/ARP.EXE
	file length: 4406784 , Exceed max length ! Ignored !
	original score: 0.0011175961699336767

FILE: Benign PE Samples Amber/AtBroker.exe
	file length: 4435456 , Exceed max length ! Ignored !
	original score: 0.0010537263005971909



FILE: Benign PE Samples Amber/BackgroundTransferHost.exe
	file length: 4427264 , Exceed max length ! Ignored !
	original score: 0.0005593677051365376

FILE: Benign PE Samples Amber/ByteCodeGenerator.exe
	file length: 4435456 , Exceed max length ! Ignored !
	original score: 0.0011135950917378068



FILE: Benign PE Samples Amber/CameraSettingsUIHost.exe
	file length: 4410880 , Exceed max length ! Ignored !
	original score: 0.00040163355879485607

FILE: Benign PE Samples Amber/CheckNetIsolation.exe
	file length: 4414976 , Exceed max length ! Ignored !
	original score: 0.0020342920906841755



FILE: Benign PE Samples Amber/CloudNotifications.exe
	file length: 4455936 , Exceed max length ! Ignored !
	original score: 0.0007048525149002671

FILE: Benign PE Samples Amber/ComputerDefaults.exe
	file length: 4447744 , Exceed max length ! Ignored !
	original score: 0.0018510161899030209



FILE: Benign PE Samples Amber/HOSTNAME.EXE
	file length: 4402688 , Exceed max length ! Ignored !
	original score: 0.0005855709314346313

FILE: Benign PE Samples Amber/InfDefaultInstall.exe
	file length: 4402688 , Exceed max length ! Ignored !
	original score: 0.0005167473573237658

FILE: Benign PE Samples UPX/ARP.EXE
	file length: 13824
	pad length: 1382
	loss: nan
	score: nan
	original score: 0.8495808839797974



FILE: Benign PE Samples UPX/AtBroker.exe
	file length: 25088
	pad length: 2508
	loss: 0.10764212906360626
	score: 0.10764212906360626
	original score: 0.10764212906360626

FILE: Benign PE Samples UPX/BackgroundTransferHost.exe
	file length: 15872
	pad length: 1587
	loss: nan
	score: nan
	original score: 0.883776843547821

FILE: Benign PE Samples UPX/ByteCodeGenerator.exe
	file length: 28160
	pad length: 2816
	loss: nan
	score: nan
	original score: 0.9686155915260315

FILE: Benign PE Samples UPX/CameraSettingsUIHost.exe
	file length: 22296
	pad length: 2229
	loss: nan
	score: nan
	original score: 0.9980714321136475

FILE: Benign PE Samples UPX/CheckNetIsolation.exe
	file length: 14848
	pad length: 1484
	loss: nan
	score: nan
	original score: 0.9417656064033508

FILE: Benign PE Samples UPX/CloudNotifications.exe
	file length: 40616
	pad length: 4061
	loss: nan
	score: nan
	original score: 0.9023998975753784

FILE: Benign PE Samples UPX/ComputerDefaults.exe
	file length: 48640
	pad lengt

### 4. 결과 로그 확인

생성 과정에서 기록한 로그를 표로 확인합니다. `original score` 와 `predict score` 를 비교하면, 공격이 적용된 파일의 탐지 점수가 "정상" 쪽(값이 큼)으로 이동한 것을 볼 수 있습니다.

In [5]:
pd.read_csv(log_path)

,filename,original score,file length,pad length,loss,predict score
0,ARP.EXE,0.612477,22528,2252,NaN,NaN
1,AtBroker.exe,0.039519,49152,4915,0.039519,0.039519
2,BackgroundTransferHost.exe,0.994638,34816,3481,NaN,NaN
3,ByteCodeGenerator.exe,0.994794,48128,4812,NaN,NaN
4,CameraSettingsUIHost.exe,0.999973,29464,2946,NaN,NaN
5,CheckNetIsolation.exe,0.943016,26112,2611,NaN,NaN
6,CloudNotifications.exe,0.978984,66728,6672,NaN,NaN
7,ComputerDefaults.exe,0.014175,61440,6144,0.014175,0.014175
8,HOSTNAME.EXE,0.999435,11776,1177,NaN,NaN
9,InfDefaultInstall.exe,0.309774,11776,1177,0.999692,1.000000
